# From 1–2 seats to 5+: what signals growth?

**Question.** Of the clients that hold 1 or 2 seats, which go on to hold **5 or more**? What do they look like beforehand, and which signals are missing in the ones that never grow?

**Context.** `stable_seat_churn.ipynb` (Part 3B) showed that growth from 1 seat is rare and slow. About 1 in 10 clients that start with 1 seat reach 5+ within 4 years, and the strongest early tell is a 2nd seat within the first year. This notebook asks the wider question for every 1–2 seat client-month, using every source in this folder.

**Sources.**
| Source | Used for | Years |
|---|---|---|
| SC Engagement List (15 Nov 2026) | seats, hires, exits, replacements, staff tenure | 2018+ snapshots |
| Staff Central *Staff Report with Details* + `shared/jp_titles.py` | JP title / category of the seats | person's role today |
| MyStaff (11 Sep 2026) | region, industry, billing currency, lead source | client attributes today |
| PB-4108 (UB4108 tickets), NCNS, Tardiness, Absences, Late Payments, Client NPS | exit reasons, attendance, payments, satisfaction | 2023+ snapshots only (step 12) |

**Approach.**
1. Build the monthly seat count per client (same cleaning and seat rules as `stable_seat_churn.ipynb`).
2. **Snapshot** = a client at a month-end holding **1 or 2 seats** that has **never held 5+ seats before**.
3. **Outcome** = the client reaches **5+ seats at any month-end in the next 36 months**. Only snapshots with 36 months of data after them are used. A 24-month version is used for the 2023+ sources.
4. Describe each snapshot using only what was known at that month-end. Compare growers and non-growers **one signal at a time** ("× as likely" than the typical 1–2 seat client), then **all signals together** in two models.

A client appears in many monthly snapshots, so every interval and test below resamples or splits **by client**, never by row. Outputs show counts only, never client names or shortcodes. All assumptions are listed in the last cell.

In [ ]:
# 1 · Setup
import re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)

DATA = Path.home() / 'Desktop/Projects/Churn Analysis/Data'
ENGAGEMENTS = DATA / 'SC Engagement List2026-11-15 09_48_04 (1).xlsx'
STAFF_REPORT = Path.home() / 'Desktop/Projects/Data/Top_roles_APAC_adhoc/Staff Report with Details 16-Nov-2026 [CONFIDENTIAL] (1).xlsx'
MYSTAFF = Path.home() / 'Desktop/Projects/Data/Top_roles_APAC_adhoc/CONFIDENTIAL Mystaff 11092026 - Copy.xlsx'
OUT_DIR = Path.home() / 'Desktop/Projects/Churn Analysis/Outputs'      # client-level files go here, never into the repo

LAST_MONTH = pd.Period('2026-10', 'M')          # last complete month-end before the 15 Nov 2026 extract
SNAP_FROM = pd.Period('2018-01', 'M')           # earliest snapshot
SMALL = (1, 2)                                  # seat counts that make a snapshot
TARGET = 5                                      # "grew" = reached this many seats
H = 36                                          # months ahead for the main outcome
H_RECENT = 24                                   # months ahead for the 2023+ sources
EXIT_CONFIRM_MONTHS = 3                         # a drop to 0 is churn only if it stays 0 this many month-ends
REPLACE_WINDOW_DAYS = 62                        # a start within this many days of an exit = a replacement
BOOT = 300; SEED = 7
C_UP, C_DOWN, C_GREY = '#00AB9C', '#F46600', '#8E8E93'

REPO_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').is_file())
JP_DIR = next(d for d in [REPO_ROOT / 'reference_code' / 'JP_titles_cat_cleaning', REPO_ROOT / 'shared'] if (d / 'jp_titles.py').is_file())
sys.path.insert(0, str(JP_DIR))                 # jp_titles.py moved from shared/ to reference_code/JP_titles_cat_cleaning/
from jp_titles import backfill_jp

def months_between(a, b):
    # whole months from Period a to Period b (b - a)
    return b.ordinal - a.ordinal if hasattr(a, 'ordinal') else (b.map(lambda p: p.ordinal) - a.map(lambda p: p.ordinal))

In [ ]:
# 2 · Load and clean the engagement list (same rules as stable_seat_churn.ipynb step 2)
raw = pd.read_excel(ENGAGEMENTS, dtype=str)
qa = []
def step(label, n): qa.append({'step': label, 'rows': int(n)})
step('Source rows', len(raw))
df = raw.copy()
df['start'] = pd.to_datetime(df['Engagement Date'], errors='coerce')
df['end'] = pd.to_datetime(df['Disengagement Date'], errors='coerce')
drop = df['Engagement ID'].duplicated(keep='first'); step('Dropped: duplicate Engagement ID (first kept)', drop.sum()); df = df[~drop]
INTERNAL_NAME = re.compile(r'cloudstaff|staff pool|internal business|pd training', re.I)
internal = df['Shortcode'].fillna('').str.upper().str.startswith('CS') | df['Client Name'].fillna('').str.contains(INTERNAL_NAME)
step('Dropped: internal or test accounts', internal.sum()); df = df[~internal]
drop = df['Shortcode'].isna(); step('Dropped: no shortcode', drop.sum()); df = df[~drop]
drop = df['Engagement Status'].eq('bench'); step('Dropped: engagement status "bench"', drop.sum()); df = df[~drop]
drop = df['start'].isna(); step('Dropped: no engagement date', drop.sum()); df = df[~drop]
drop = df['end'].notna() & (df['end'] < df['start']); step('Dropped: disengagement before engagement', drop.sum()); df = df[~drop]
drop = df['Engagement Status'].eq('disengage') & df['end'].isna(); step('Dropped: status "disengage" but no date', drop.sum()); df = df[~drop]
step('Engagements used (external client seats)', len(df))
eng = df[['Engagement ID', 'CSID', 'Shortcode', 'start', 'end']].rename(columns={'Shortcode': 'client'}).reset_index(drop=True)
print(f"{eng['client'].nunique():,} external clients · engagements from {eng['start'].min():%b %Y} to {eng['start'].max():%b %Y}")
pd.DataFrame(qa)

In [ ]:
# 3 · Monthly seat count per client (a seat counts at a month-end if the engagement is active that day)
starts = pd.DataFrame({'client': eng['client'], 'month': eng['start'].dt.to_period('M'), 'delta': 1})
ended = eng[eng['end'].notna()]
end_month = ended['end'].dt.to_period('M')
end_month = end_month.where(~ended['end'].dt.is_month_end, end_month + 1)     # ends on the last day still count that month
ends = pd.DataFrame({'client': ended['client'], 'month': end_month, 'delta': -1})
events = pd.concat([starts, ends], ignore_index=True)
events = events[events['month'] <= LAST_MONTH]

net = events.groupby(['client', 'month'])['delta'].sum()
frames = []
for client, s in net.groupby(level=0):
    s = s.droplevel(0)
    idx = pd.period_range(s.index.min(), LAST_MONTH, freq='M')
    frames.append(pd.DataFrame({'client': client, 'month': idx, 'seats': s.reindex(idx, fill_value=0).cumsum().to_numpy()}))
panel = pd.concat(frames, ignore_index=True).sort_values(['client', 'month']).reset_index(drop=True)
panel['t'] = panel['month'].map(lambda p: p.ordinal)

# check against a direct count of active engagements on two month-ends
for m in [pd.Period('2021-06', 'M'), LAST_MONTH]:
    me = m.to_timestamp(how='end').normalize()
    direct = eng[(eng['start'] <= me) & (eng['end'].isna() | (eng['end'] >= me))].groupby('client').size()
    fe = panel[(panel['month'] == m) & (panel['seats'] > 0)].set_index('client')['seats']
    assert direct.sort_index().equals(fe.sort_index().astype(direct.dtype)), m
assert (panel['seats'] >= 0).all()

# confirmed churn-out: seats go from >0 to 0 and stay 0 for EXIT_CONFIRM_MONTHS month-ends (or until the data ends)
g = panel.groupby('client', sort=False)
prev = g['seats'].shift().fillna(0)
zeros_ahead = []
for _, s in g['seats']:
    a = s.to_numpy(); z = np.zeros(len(a), dtype=int)
    for i in range(len(a) - 1, -1, -1):
        z[i] = (z[i + 1] + 1 if i + 1 < len(a) else 1) if a[i] == 0 else 0
    zeros_ahead.append(z)
panel['zeros_ahead'] = np.concatenate(zeros_ahead)
panel['exit'] = (panel['seats'] == 0) & (prev > 0) & (panel['zeros_ahead'] >= np.minimum(EXIT_CONFIRM_MONTHS, LAST_MONTH.ordinal - panel['t'] + 1))
print(f"{len(panel):,} client-months · {panel.loc[panel.seats > 0, 'client'].nunique():,} clients ever active · checks passed")

In [ ]:
# 4 · Snapshots and outcome
#     snapshot: month-end with 1-2 seats, from SNAP_FROM, client never had TARGET+ seats before (or at) that month-end
#     outcome:  seats >= TARGET at any month-end in the next H months (also: next H_RECENT months; reached 3+; churned out)
p = panel.copy()
g = p.groupby('client', sort=False)
p['peak_so_far'] = g['seats'].cummax()
p['first_t'] = p['client'].map(p[p.seats > 0].groupby('client')['t'].min())

def future_max(s, h):
    # max of the next h values (t+1 .. t+h), per client; NaN where fewer than h months remain
    r = s[::-1].rolling(h, min_periods=h).max()[::-1]
    return r.shift(-1)
for hh in sorted({H, H_RECENT, 12}):
    p[f'fmax_{hh}'] = g['seats'].transform(lambda s: future_max(s, hh))
# months until the first month-end at TARGET+ (any time later)
nxt = []
for _, s in g:
    tt, hit = s['t'].to_numpy(), (s['seats'] >= TARGET).to_numpy()
    out = np.full(len(tt), np.nan); upcoming = np.nan
    for i in range(len(tt) - 1, -1, -1):
        out[i] = upcoming - tt[i] if not np.isnan(upcoming) else np.nan
        if hit[i]: upcoming = tt[i]
    nxt.append(out)
p['months_to_target'] = np.concatenate(nxt)
exit_t = p[p['exit']].groupby('client')['t'].apply(np.array)
def exit_within(row_client, t, h):
    a = exit_t.get(row_client)
    return bool(a is not None and ((a > t) & (a <= t + h)).any())

snap_all = p[p['seats'].isin(SMALL) & (p['month'] >= SNAP_FROM) & (p['peak_so_far'] < TARGET)].copy()
snap_all['grew'] = (snap_all[f'fmax_{H}'] >= TARGET).astype(float).where(snap_all[f'fmax_{H}'].notna())
snap_all['grew_recent'] = (snap_all[f'fmax_{H_RECENT}'] >= TARGET).astype(float).where(snap_all[f'fmax_{H_RECENT}'].notna())
snap_all['grew3'] = (snap_all[f'fmax_{H}'] >= 3).astype(float).where(snap_all[f'fmax_{H}'].notna())
snap_all['churned'] = [exit_within(c, t, H) for c, t in zip(snap_all['client'], snap_all['t'])]

n_former = p[p['seats'].isin(SMALL) & (p['month'] >= SNAP_FROM) & (p['peak_so_far'] >= TARGET)]
snap = snap_all[snap_all['grew'].notna()].copy()
snap['y'] = snap['grew'].astype(int)
print(f"1-2 seat client-months from {SNAP_FROM}: {len(snap_all) + len(n_former):,}. Left out: {len(n_former):,} from clients that already had {TARGET}+ seats before "
      f"({n_former.client.nunique():,} clients: shrinking big clients, a different story).")
print(f"With {H} months observed: {len(snap):,} snapshots from {snap.client.nunique():,} clients, {snap.month.min()} to {snap.month.max()}.")

def client_rate(d, col='y'):
    # share of clients with at least one positive snapshot
    return d.groupby('client')[col].max().mean()
summary = pd.DataFrame({
    'snapshots': snap.groupby('seats').size(),
    'clients': snap.groupby('seats')['client'].nunique(),
    f'reached {TARGET}+ within {H} m (share of snapshots)': snap.groupby('seats')['y'].mean().map('{:.1%}'.format),
    f'reached 3+ within {H} m': snap.groupby('seats')['grew3'].mean().map('{:.0%}'.format),
    f'churned out within {H} m': snap.groupby('seats')['churned'].mean().map('{:.0%}'.format),
})
summary.loc['1-2 (all)'] = [len(snap), snap.client.nunique(), f"{snap.y.mean():.1%}", f"{snap.grew3.mean():.0%}", f"{snap.churned.mean():.0%}"]
summary.index.name = 'seats at snapshot'
print(f"Clients: {snap.groupby('client').y.max().sum():,} of {snap.client.nunique():,} ({client_rate(snap):.0%}) reached {TARGET}+ from a 1-2 seat snapshot.")
summary

In [ ]:
# 5 · Growth rate by snapshot year (does the base rate move over time?)
by_year = snap.groupby(snap['month'].dt.year).agg(snapshots=('y', 'size'), clients=('client', 'nunique'), rate=('y', 'mean'), rate3=('grew3', 'mean'), churn=('churned', 'mean'))
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(by_year.index, by_year['rate'], marker='o', color=C_UP, lw=2, label=f'reached {TARGET}+')
ax.plot(by_year.index, by_year['rate3'], marker='o', color='#007AFF', lw=1.5, label='reached 3+')
ax.plot(by_year.index, by_year['churn'], marker='o', color=C_DOWN, lw=1.5, ls='--', label='churned out')
for x, v in by_year['rate'].items(): ax.annotate(f'{v:.0%}', (x, v), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=8, color=C_UP)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_ylim(0, None); ax.grid(alpha=.3)
ax.set_title(f'1-2 seat snapshots: what happened in the next {H} months, by snapshot year'); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); plt.show()
by_year.assign(rate=by_year.rate.map('{:.1%}'.format), rate3=by_year.rate3.map('{:.0%}'.format), churn=by_year.churn.map('{:.0%}'.format)).rename(
    columns={'rate': f'reached {TARGET}+', 'rate3': 'reached 3+', 'churn': 'churned out'})

## Signals at each snapshot

Everything below is measured **at the snapshot month-end, from data up to that day**. Nothing from after the snapshot is used.

| Group | Signal |
|---|---|
| Size and stage | seats now (1 or 2), months since the first seat, seats vs 6 months earlier, months at the current count, past peak (3–4), months since the last seat was added |
| Hiring and staff | seats added and staff exits in the last 12 months, exits **replaced** (a new start within 2 months), average months current staff have been in their seats, started with 2+ seats in the first month, came back after an earlier churn-out |
| Roles | JP category and title base of the current seats (`jp_titles`, person's role today); for 2 seats, whether both are the same role |
| Client | MyStaff region, industry roll-up, billing currency, lead source |

In [ ]:
# 6 · Size, stage, hiring and staff signals (engagement list only)
s = snap.copy()
s['age_m'] = s['t'] - s['first_t'] + 1
pp = p.set_index(['client', 't'])['seats']
def lag(df_, k):
    return pd.Series(pp.reindex(list(zip(df_['client'], df_['t'] - k))).to_numpy(), index=df_.index)
s['seats_6m_ago'] = lag(s, 6)
s['change_6m'] = (s['seats'] - s['seats_6m_ago']).where(s['age_m'] > 6)
# months at the current count
run = p.groupby(['client', (p['seats'] != p.groupby('client')['seats'].shift()).cumsum()]).cumcount() + 1
s['run_m'] = run.reindex(s.index).to_numpy()
s['past_peak'] = s['peak_so_far']

# engagement-level history at each snapshot: vectorised per client
s['month_end'] = s['month'].dt.to_timestamp(how='end').dt.normalize()
e = eng.copy()
e['end_f'] = e['end'].fillna(pd.Timestamp('2100-01-01'))
# replacement: an exit followed (or preceded) by another start at the same client within REPLACE_WINDOW_DAYS
st_by_client = e.groupby('client')['start'].apply(lambda x: np.sort(x.to_numpy()))
def replaced(row):
    if pd.isna(row['end']): return False
    a = st_by_client[row['client']]
    d = (a - np.datetime64(row['end'])) / np.timedelta64(1, 'D')
    return bool(((d >= -REPLACE_WINDOW_DAYS) & (d <= REPLACE_WINDOW_DAYS) & (a != np.datetime64(row['start']))).any())
e['replaced'] = e.apply(replaced, axis=1)
first_month_starts = e.assign(m=e['start'].dt.to_period('M')).groupby('client').apply(lambda d: (d['m'] == d['m'].min()).sum(), include_groups=False)

rows = []
for c, d in s.groupby('client', sort=False):
    ec = e[e['client'] == c]
    st, en, rp = ec['start'].to_numpy(), ec['end_f'].to_numpy(), ec['replaced'].to_numpy()
    for idx, me in zip(d.index, d['month_end'].to_numpy()):
        y1 = me - np.timedelta64(365, 'D')
        active = (st <= me) & (en >= me)
        ex12 = (en > y1) & (en <= me)
        past_starts = st[st <= me]
        rows.append((idx, int(((st > y1) & (st <= me)).sum()), int(ex12.sum()), int((ex12 & rp).sum()),
                     float(((me - st[active]) / np.timedelta64(1, 'D')).mean() / 30.44) if active.any() else np.nan,
                     float((me - past_starts.max()) / np.timedelta64(1, 'D') / 30.44) if len(past_starts) else np.nan,
                     int(((st <= me) & (en < me)).sum())))
h = pd.DataFrame(rows, columns=['idx', 'starts_12m', 'exits_12m', 'replaced_12m', 'staff_tenure_m', 'months_since_add', 'staff_exits_ever']).set_index('idx')
s = s.join(h)
s['prior_churn'] = [bool((exit_t.get(c) is not None) and (exit_t[c] < t).any()) for c, t in zip(s['client'], s['t'])]
s['first_month_seats'] = s['client'].map(first_month_starts)
s['snap_year'] = s['month'].dt.year
print(f"{len(s):,} snapshots with size, stage, hiring and staff signals")
s[['seats', 'age_m', 'change_6m', 'run_m', 'past_peak', 'starts_12m', 'exits_12m', 'replaced_12m', 'staff_tenure_m', 'months_since_add', 'first_month_seats']].describe().T.round(1)

In [ ]:
# 7 · Role signals: JP category / title base of the seats held at the snapshot (staff report, cleaned by shared/jp_titles.py)
sr_raw = pd.read_excel(STAFF_REPORT, dtype=str)
sr, jp_reports = backfill_jp(sr_raw)
by_csid = sr.drop_duplicates('CSID', keep='last').set_index('CSID')
seat_rows = s[['client', 'month_end']].reset_index().merge(eng[['client', 'CSID', 'start', 'end']], on='client')
seat_rows = seat_rows[(seat_rows['start'] <= seat_rows['month_end']) & (seat_rows['end'].isna() | (seat_rows['end'] >= seat_rows['month_end']))]
seat_rows['cat'] = seat_rows['CSID'].map(by_csid['JP Category (Filled)']).fillna('Unknown')
seat_rows['title'] = seat_rows['CSID'].map(by_csid['JP Title Base (Filled)']).fillna('Unknown')
agg = seat_rows.groupby('index').agg(cats=('cat', lambda x: tuple(sorted(x))), titles=('title', lambda x: tuple(sorted(x))))
s = s.join(agg)
assert s['cats'].map(len).eq(s['seats']).all()
s['role_unknown'] = s['cats'].map(lambda c: 'Unknown' in c)
known2 = (s['seats'] == 2) & ~s['role_unknown']
s['same_cat_2'] = (known2 & s['cats'].map(lambda c: len(set(c)) == 1))
s['diff_cat_2'] = (known2 & s['cats'].map(lambda c: len(set(c)) == 2))
s['same_title_2'] = ((s['seats'] == 2) & s['titles'].map(lambda c: 'Unknown' not in c and len(set(c)) == 1))
CAT_MIN = 300                                   # snapshots holding a seat in that category, to get its own signal
cat_counts = pd.Series([c for cs in s['cats'] for c in set(cs) if c != 'Unknown']).value_counts()
TOP_CATS = cat_counts[cat_counts >= CAT_MIN].index.tolist()
for c in TOP_CATS:
    s[f'cat::{c}'] = s['cats'].map(lambda cs, c=c: c in cs)
print(f"Role known for every seat in {(~s.role_unknown).mean():.0%} of snapshots. Categories with their own signal ({CAT_MIN}+ snapshots): {len(TOP_CATS)}")
cat_counts.head(20).rename('snapshots with a seat in this category').to_frame()

In [ ]:
# 8 · Client attributes from MyStaff (attributes as they are today)
ms = pd.read_excel(MYSTAFF, sheet_name='MyStaff', usecols=['Shortcode', 'Source', 'Billing Currency', 'Region', 'Industry Roll Up'], dtype=str)
ms = ms.dropna(subset=['Shortcode']).drop_duplicates('Shortcode').set_index('Shortcode')
ms = ms.replace({'-': np.nan, '': np.nan})
for col, new in [('Region', 'region'), ('Industry Roll Up', 'industry'), ('Billing Currency', 'currency'), ('Source', 'source')]:
    s[new] = s['client'].map(ms[col]).fillna('Not in MyStaff' if col == 'Region' else 'Unknown')
cov_ms = s.groupby('client').agg(y=('y', 'max'), churned=('churned', 'max'), found=('region', lambda x: (x != 'Not in MyStaff').any()))
print(f"MyStaff rows: {len(ms):,}. Snapshot clients found in MyStaff: {cov_ms.found.mean():.0%} "
      f"(clients that grew: {cov_ms[cov_ms.y == 1].found.mean():.0%}; that churned out: {cov_ms[cov_ms.churned].found.mean():.0%}). "
      "MyStaff covers recent clients better, so its signals are read with that in mind.")
pd.concat({c: s.groupby(c)['client'].nunique().sort_values(ascending=False).head(10) for c in ['region', 'industry', 'currency', 'source']}).rename('clients').to_frame()

## One signal at a time

For each signal: the share of snapshots that reached 5+ within 36 months **when the signal is present**, and how that compares with the **typical 1–2 seat client** (all snapshots). **2× = twice as likely to grow**, 0.5× = half as likely. The 95% intervals come from resampling **clients**, not rows.

- Signals **above 1×** point towards growth.
- Signals **below 1×** are the *lack of indicators*: conditions under which growth almost never happens.
- An interval that crosses 1× is not a clear signal either way.

In [ ]:
# 9 · Signal table: "x as likely to reach 5+ as the typical 1-2 seat client", with client-resampled 95% intervals
def flag_defs(d):
    F = {
        'Size and stage': {
            '2 seats now (vs 1)': d['seats'] == 2,
            'New client: first seat under 12 months ago': d['age_m'] <= 12,
            'Client for 3+ years': d['age_m'] > 36,
            'Grew in the last 6 months': d['change_6m'] > 0,
            'Shrank in the last 6 months': d['change_6m'] < 0,
            'Same seat count for 12+ months': d['run_m'] >= 12,
            'Was bigger before (peak 3-4 seats)': d['past_peak'] >= 3,
            'Added a seat in the last 3 months': d['months_since_add'] <= 3,
            'No seat added in the last 2 years': d['months_since_add'] > 24,
        },
        'Hiring and staff': {
            '2+ seats added in the last 12 months': d['starts_12m'] >= 2,
            'No seat added in the last 12 months': d['starts_12m'] == 0,
            'Any staff exit in the last 12 months': d['exits_12m'] > 0,
            'An exit was replaced (new start within 2 months)': d['replaced_12m'] > 0,
            'An exit was not replaced': d['exits_12m'] > d['replaced_12m'],
            'Current staff in seat under 6 months on average': d['staff_tenure_m'] < 6,
            'Current staff in seat 24+ months on average': d['staff_tenure_m'] >= 24,
            'Started with 2+ seats in its first month': d['first_month_seats'] >= 2,
            'Came back after an earlier churn-out': d['prior_churn'],
        },
        'Roles': {
            'Any seat role unknown': d['role_unknown'],
            '2 seats in the same JP category': d['same_cat_2'],
            '2 seats in different JP categories': d['diff_cat_2'],
            '2 seats with the same title': d['same_title_2'],
            **{f'Seat in {c}': d[f'cat::{c}'] for c in TOP_CATS},
        },
    }
    return F

def mystaff_flags(d):
    return {'Client (MyStaff, clients in MyStaff only)': {
        **{f'Region: {r}': d['region'] == r for r in ['APAC', 'AMER']},
        **{f'Billed in {c}': d['currency'] == c for c in ['AUD', 'USD']},
        **{f'Industry: {c}': d['industry'] == c for c in d['industry'].value_counts().loc[lambda v: ~v.index.isin(['Unknown', 'CLOUDSTAFF'])].head(8).index},
    }}

def signal_table(d, F, ycol='y', boot=BOOT, min_clients=15, seed=SEED):
    rng = np.random.default_rng(seed)
    clients, cidx = np.unique(d['client'].to_numpy(), return_inverse=True)
    W = rng.poisson(1.0, size=(boot, len(clients)))                       # client bootstrap weights
    y = d[ycol].to_numpy(float)
    ny, nn = np.bincount(cidx, weights=y, minlength=len(clients)), np.bincount(cidx, minlength=len(clients)).astype(float)
    base_b = (W @ ny) / (W @ nn)
    base = y.mean()
    out = []
    for grp, flags in F.items():
        for name, f in flags.items():
            f = f.fillna(False).to_numpy(bool)
            if f.sum() == 0: continue
            fy = np.bincount(cidx, weights=y * f, minlength=len(clients)); fn = np.bincount(cidx, weights=f.astype(float), minlength=len(clients))
            n_cl = int((fn > 0).sum())
            if n_cl < min_clients: continue
            with np.errstate(invalid='ignore', divide='ignore'):
                rb = ((W @ fy) / (W @ fn)) / base_b
            r = y[f].mean()
            out.append({'group': grp, 'signal': name, 'snapshots': int(f.sum()), 'clients': n_cl, 'clients that grew': int((fy > 0).sum()),
                        'growth rate': r, 'x typical': r / base, 'lo': np.nanpercentile(rb, 2.5), 'hi': np.nanpercentile(rb, 97.5),
                        'share of snapshots': f.mean()})
    t = pd.DataFrame(out)
    t['reading'] = np.select([t['lo'] > 1, t['hi'] < 1], ['towards growth', 'against growth'], 'not clear')
    return t, base

F = flag_defs(s)
sig, base = signal_table(s, F)
print(f"Typical 1-2 seat client: {base:.1%} of snapshots reach {TARGET}+ within {H} months. Signals shown when at least 15 clients have them.")

def plot_signals(t, title, base_rate):
    t = t.sort_values('x typical')
    fig, ax = plt.subplots(figsize=(10, 0.3 * len(t) + 1.4))
    ys = np.arange(len(t))
    col = t['reading'].map({'towards growth': C_UP, 'against growth': C_DOWN, 'not clear': C_GREY}).to_numpy()
    for y_, (_, r), c_ in zip(ys, t.iterrows(), col):
        lo, hi = max(r['lo'], 0.02), min(r['hi'], 50)
        ax.plot([lo, hi], [y_, y_], color=c_, alpha=.6, lw=2)
    ax.scatter(t['x typical'].clip(0.02, 50), ys, c=col, zorder=3, s=28)
    ax.axvline(1, color='black', lw=.8)
    ax.set_xscale('log'); ax.set_xticks([0.05, 0.1, 0.25, 0.5, 1, 2, 4, 8, 16], ['0.05x', '0.1x', '0.25x', '0.5x', '1x', '2x', '4x', '8x', '16x'])
    SHORT = {'Size and stage': 'stage', 'Hiring and staff': 'staff', 'Roles': 'role'}
    ax.set_yticks(ys, [f"{n}  ({SHORT.get(g, g.split(' (')[0].lower())})" for n, g in zip(t['signal'], t['group'])], fontsize=8.5)
    ax.set_xlabel(f'times as likely to reach {TARGET}+ as the typical 1-2 seat client ({base_rate:.1%}); log scale, 95% interval')
    ax.set_title(title); ax.grid(axis='x', alpha=.3)
    plt.tight_layout(); plt.show()

plot_signals(sig, f'Signals at a 1-2 seat snapshot and the chance of reaching {TARGET}+ seats within {H} months', base)
def show_sig(t):
    o = t.sort_values(['group', 'x typical'], ascending=[True, False]).copy()
    o['growth rate'] = o['growth rate'].map('{:.1%}'.format); o['share of snapshots'] = o['share of snapshots'].map('{:.0%}'.format)
    o['x typical (95%)'] = o.apply(lambda r: f"{r['x typical']:.1f}x ({r.lo:.1f}-{r.hi:.1f})", axis=1)
    return o[['group', 'signal', 'share of snapshots', 'clients', 'clients that grew', 'growth rate', 'x typical (95%)', 'reading']].set_index(['group', 'signal'])
show_sig(sig)

In [ ]:
# 9b · Client attributes (MyStaff), compared only among clients that are in MyStaff
#      Outside MyStaff the snapshot client is mostly a past client, so "in MyStaff" would just mean "still here today".
in_ms = s[s['region'] != 'Not in MyStaff']
sig_ms, base_ms = signal_table(in_ms, mystaff_flags(in_ms), min_clients=15)
print(f"Clients in MyStaff: {in_ms.client.nunique():,}; their typical growth rate is {base_ms:.1%} (vs {base:.1%} for all clients), so 1x here means {base_ms:.1%}.")
show_sig(sig_ms)

In [ ]:
# 10 · The 2nd seat: how fast it came, for snapshots at 2 seats (an earlier notebook found this was the strongest early tell)
two = s[s['seats'] == 2].copy()
second = p[p.seats >= 2].groupby('client')['t'].min()
two['months_to_2nd'] = two['client'].map(second) - two['first_t'] + 1
two['2nd seat came'] = pd.cut(two['months_to_2nd'], [-1, 1, 6, 12, 24, 10**4], labels=['with the 1st (same month)', 'within 6 months', '7-12 months', '13-24 months', 'after 2+ years'])
t2 = two.groupby('2nd seat came', observed=True).agg(snapshots=('y', 'size'), clients=('client', 'nunique'), grew=('y', 'mean'))
t2[f'reached {TARGET}+ within {H} m'] = t2.pop('grew').map('{:.1%}'.format)
one = s[s['seats'] == 1]
print(f"Reference: 1-seat snapshots reach {TARGET}+ at {one.y.mean():.1%}, 2-seat snapshots at {two.y.mean():.1%}.")
t2

## All signals together

Many single signals overlap. New clients are also the ones adding seats, and an unreplaced exit is also a shrink. Two models use every signal at once, to show **which ones still matter when the others are known**:
- **Logistic regression** on standardised signals: one readable weight per signal.
- **LightGBM** (gradient-boosted trees): catches thresholds and interactions; its explanations come from SHAP values.

How well they separate growers from non-growers is tested two ways:
1. **New clients**: 5-fold cross-validation where a client is never in both the training and the test folds.
2. **Later years**: trained on snapshots up to 2021, tested on 2022–2023.

AUC 0.5 is a coin flip and about 0.7 is useful.

In [ ]:
# 11 · Models: logistic regression and LightGBM, client-grouped CV and out-of-time test
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import shap

NUM = {   # model feature -> label
    'seats': 'Seats now (1 or 2)', 'age_m': 'Months since first seat', 'change_6m': 'Seats vs 6 months earlier',
    'run_m': 'Months at current seat count', 'past_peak': 'Past peak seat count', 'months_since_add': 'Months since a seat was added',
    'starts_12m': 'Seats added, last 12 months', 'exits_12m': 'Staff exits, last 12 months', 'replaced_12m': 'Exits replaced, last 12 months',
    'staff_tenure_m': 'Avg months current staff in seat', 'first_month_seats': 'Seats started in the first month',
    'prior_churn': 'Came back after a churn-out', 'role_unknown': 'Seat role unknown', 'same_cat_2': '2 seats, same JP category',
    'snap_year': 'Snapshot year',
}
X = s[list(NUM)].astype(float).copy()
X['change_6m'] = X['change_6m'].fillna(0); X['staff_tenure_m'] = X['staff_tenure_m'].fillna(0); X['months_since_add'] = X['months_since_add'].fillna(X['age_m'])
for i, c in enumerate(TOP_CATS):                       # safe column names for LightGBM
    X[f'cat_{i}'] = s[f'cat::{c}'].astype(float); NUM[f'cat_{i}'] = f'Seat in {c}'
# MyStaff attributes are left out of the models: being in MyStaff mostly means "still a client today" (step 8)
Xm = X.drop(columns=['snap_year'])                      # year is a control in the OOT test only via the split, not a signal
yv, groups = s['y'].to_numpy(), s['client'].to_numpy()

def make_lr(): return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000, class_weight='balanced'))
def make_gbm(): return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=60, subsample=0.8, subsample_freq=1,
                                          colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, random_state=SEED)
res = []
oof = {k: np.zeros(len(s)) for k in ('Logistic regression', 'LightGBM')}
for tr, te in GroupKFold(n_splits=5).split(Xm, yv, groups):
    for k, mk in (('Logistic regression', make_lr), ('LightGBM', make_gbm)):
        oof[k][te] = mk().fit(Xm.iloc[tr], yv[tr]).predict_proba(Xm.iloc[te])[:, 1]
for k in oof:
    res.append({'model': k, 'test': 'new clients (5-fold, by client)', 'AUC': roc_auc_score(yv, oof[k]), 'test snapshots': len(s), 'test growers (clients)': int(s.groupby('client').y.max().sum())})
tr_m, te_m = s['snap_year'] <= 2021, s['snap_year'] >= 2022
for k, mk in (('Logistic regression', make_lr), ('LightGBM', make_gbm)):
    pr = mk().fit(Xm[tr_m], yv[tr_m]).predict_proba(Xm[te_m])[:, 1]
    res.append({'model': k, 'test': 'later years (train <=2021, test 2022-23)', 'AUC': roc_auc_score(yv[te_m], pr), 'test snapshots': int(te_m.sum()),
                'test growers (clients)': int(s[te_m].groupby('client').y.max().sum())})
res = pd.DataFrame(res); res['AUC'] = res['AUC'].round(3)

# how much do the top-scored snapshots concentrate growth? (cross-validated LightGBM)
q = pd.qcut(pd.Series(oof['LightGBM']).rank(method='first'), 10, labels=False)
lift = pd.DataFrame({'y': yv, 'decile': 10 - q.to_numpy()}).groupby('decile')['y'].mean()
print(f"Top 10% of snapshots by cross-validated LightGBM score: {lift.loc[1]:.1%} reach {TARGET}+ ({lift.loc[1] / yv.mean():.1f}x typical). "
      f"Bottom half: {pd.Series(yv)[(10 - q.to_numpy()) > 5].mean():.1%}.")
res.set_index(['model', 'test'])

In [ ]:
# 11b · What the models lean on: logistic weights (standardised) and LightGBM SHAP (mean |SHAP| and direction)
lr_full = make_lr().fit(Xm, yv)
coef = pd.Series(lr_full[-1].coef_[0], index=Xm.columns)
gbm_full = make_gbm().fit(Xm, yv)
sv = shap.TreeExplainer(gbm_full).shap_values(Xm)
sv = sv[1] if isinstance(sv, list) else sv
imp = pd.Series(np.abs(sv).mean(0), index=Xm.columns)
direction = pd.Series([np.corrcoef(Xm[c], sv[:, i])[0, 1] if Xm[c].std() > 0 else 0 for i, c in enumerate(Xm.columns)], index=Xm.columns)
expl = pd.DataFrame({'signal': [NUM[c] for c in Xm.columns], 'LightGBM share of impact': imp / imp.sum(),
                     'LightGBM: higher value pushes growth': np.where(direction > 0.1, 'up', np.where(direction < -0.1, 'down', 'mixed')),
                     'logistic weight (per 1 sd)': coef}).sort_values('LightGBM share of impact', ascending=False)

top = expl.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(top['signal'], top['LightGBM share of impact'], color=np.where(top['LightGBM: higher value pushes growth'] == 'up', C_UP,
                                                                         np.where(top['LightGBM: higher value pushes growth'] == 'down', C_DOWN, C_GREY)))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.grid(axis='x', alpha=.3)
ax.set_title('What the LightGBM model leans on (green: higher value means more growth; orange: less)')
ax.set_xlabel('share of everything that moves the predicted chance of reaching 5+')
plt.tight_layout(); plt.show()
expl.assign(**{'LightGBM share of impact': expl['LightGBM share of impact'].map('{:.0%}'.format), 'logistic weight (per 1 sd)': expl['logistic weight (per 1 sd)'].round(2)}).set_index('signal')

In [ ]:
# 11c · Shape of the two strongest numeric signals: growth rate by band (all snapshots)
CUTS = {
    'Months since first seat': ('age_m', [0, 6, 12, 24, 36, 60, 10**4], ['0-6', '7-12', '13-24', '25-36', '37-60', '60+']),
    'Seats added, last 12 months': ('starts_12m', [-1, 0, 1, 2, 3, 100], ['0', '1', '2', '3', '4+']),
    'Months since a seat was added': ('months_since_add', [-1, 3, 6, 12, 24, 10**4], ['0-3', '4-6', '7-12', '13-24', '24+']),
    'Avg months current staff in seat': ('staff_tenure_m', [-1, 6, 12, 24, 10**4], ['under 6', '6-12', '12-24', '24+']),
}
fig, axes = plt.subplots(1, len(CUTS), figsize=(15, 3.4), sharey=True)
rows = []
for ax, (title, (col, bins, labels)) in zip(axes, CUTS.items()):
    b = pd.cut(s[col], bins, labels=labels)
    gg = s.groupby(b, observed=True).agg(rate=('y', 'mean'), n=('y', 'size'), clients=('client', 'nunique'))
    ax.bar(gg.index.astype(str), gg['rate'], color=C_UP)
    ax.axhline(base, color='black', lw=.8, ls='--')
    for i, (r, n) in enumerate(zip(gg['rate'], gg['clients'])): ax.text(i, r, f'{r:.0%}\n{n} cl.', ha='center', va='bottom', fontsize=7.5)
    ax.set_ylim(0, s.groupby(pd.cut(s[col], bins, labels=labels), observed=True)['y'].mean().max() * 1.3)
    ax.set_title(title, fontsize=10); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    for lab, r in gg.iterrows(): rows.append({'signal': title, 'band': lab, 'snapshots': int(r.n), 'clients': int(r.clients), f'reached {TARGET}+': f"{r.rate:.1%}"})
axes[0].set_ylabel(f'share reaching {TARGET}+ within {H} m')
fig.suptitle('Growth rate by band (dashed line: typical 1-2 seat client)', y=1.02); plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index(['signal', 'band'])

## 2023+ sources: exit reasons, attendance, payments, satisfaction

These files start in 2023 (UB4108 in 2020, but its reason detail is reliable only from about 2023). A 36-month outcome would leave almost no snapshots, so this layer uses snapshots from **Jan 2023 to Oct 2024** and a **24-month** outcome (reached 5+ by Oct 2026 at the latest). Each source is joined by client shortcode and month.

- **UB4108 exits, last 12 months**: any resignation, any client-initiated exit, any exit with a replacement requested.
- **Attendance, last 6 months**: any no call no show, any late arrival, any notified absence.
- **Late payments, last 6 months**: any invoice paid late.
- **Client NPS**: the latest response in the 12 months before the snapshot (promoter / passive / detractor / none).

The number of growers is small, so read this as a check for strong effects only.

In [ ]:
# 12 · 2023+ layer: build the snapshots and the extra signals
r_ = snap_all[(snap_all['month'] >= pd.Period('2023-01', 'M')) & snap_all['grew_recent'].notna()].copy()
r_['y'] = r_['grew_recent'].astype(int)
r_['month_end'] = r_['month'].dt.to_timestamp(how='end').dt.normalize()
key = lambda x: x.astype('string').str.strip().str.upper()

def recent_any(r, ev, date_col, months, mask=None, sc_col='sc'):
    # True when the client had at least one event in the `months` before the snapshot month-end (inclusive)
    ev = ev if mask is None else ev[mask]
    by = ev.groupby(sc_col)[date_col].apply(lambda x: np.sort(x.dropna().to_numpy()))
    out = []
    for c, me in zip(r['client'].str.upper(), r['month_end'].to_numpy()):
        a = by.get(c)
        out.append(bool(a is not None and ((a <= me) & (a > me - np.timedelta64(int(months * 30.44), 'D'))).any()))
    return pd.Series(out, index=r.index)

def source_end(ev, date_col):
    return ev[date_col].max()

# UB4108 tickets: reason, on the ticket's shortcode and termination notice date (submitted date if blank)
ub = pd.read_excel(DATA / 'PB-4108.xlsx', sheet_name='UB Staff Disengagement')
ub['sc'] = key(ub['Client ShortCode'] if 'Client ShortCode' in ub.columns else ub['Shortcode'])
ub['date'] = pd.to_datetime(ub['Termination Notice Date'], errors='coerce').fillna(pd.to_datetime(ub['Submitted at'].astype('string').str.upper(), format='%d-%b-%Y %I:%M%p', errors='coerce'))
dis_type = ub['Type of Disengagement'].astype('string').fillna('')
real = ~dis_type.eq('Cancelled Disengagement') & ~ub['Reason for Disengagement'].astype('string').fillna('').eq('For Record Purpose Only')
r_['ub_resignation_12m'] = recent_any(r_, ub, 'date', 12, real & ub['Reason for Disengagement'].astype('string').eq('Voluntary'))
r_['ub_client_exit_12m'] = recent_any(r_, ub, 'date', 12, real & ub['Type of Involuntary'].astype('string').eq('Customer Initiated'))
r_['ub_replacement_12m'] = recent_any(r_, ub, 'date', 12, real & dis_type.str.contains('with Replacement', case=False))

# attendance
ncns = pd.read_excel(DATA / 'NCNS.xlsx', sheet_name='NCNS'); ncns['sc'] = key(ncns['Client Shortcode']); ncns['Date'] = pd.to_datetime(ncns['Date'], errors='coerce')
tard = pd.read_excel(DATA / 'Tardiness.xlsx', sheet_name='Tardiness'); tard['sc'] = key(tard['Client Shortcode']); tard['Date'] = pd.to_datetime(tard['Date'], errors='coerce')
absn = pd.read_excel(DATA / 'Absences.xlsx', sheet_name='Absences'); absn['sc'] = key(absn['Shortcode']); absn['Date'] = pd.to_datetime(absn['Date'], errors='coerce')
r_['ncns_6m'] = recent_any(r_, ncns, 'Date', 6)
r_['tardiness_6m'] = recent_any(r_, tard, 'Date', 6)
r_['absence_6m'] = recent_any(r_, absn, 'Date', 6)

# late payments (same reshaping as churn_analysis_raw.ipynb step 8)
from datetime import datetime
lp_raw = pd.read_excel(DATA / 'Late Payments.xlsx', sheet_name=['2023', '2024', '2025', '2026'])
parts = []
for year, d0 in lp_raw.items():
    for dc in [c for c in d0.columns if isinstance(c, datetime)]:
        parts.append(pd.DataFrame({'sc': key(d0['Mystaff Shortcode']), 'date': pd.Timestamp(dc).to_period('M').to_timestamp(how='end').normalize(),
                                   'days_to_pay': d0[dc], 'late_flag': d0[f"{dc:%Y-%m-%d} 00:00:00.1"]}))
lp = pd.concat(parts, ignore_index=True); lp = lp[lp['days_to_pay'] > 0]
r_['late_payment_6m'] = recent_any(r_, lp, 'date', 6, lp['late_flag'] == 1)
r_['invoiced_6m'] = recent_any(r_, lp, 'date', 6)                      # client appears in the payment report at all

# client NPS: latest response in the 12 months before the snapshot
nps = pd.read_excel(DATA / 'Client NPS.xlsx'); nps['sc'] = key(nps['_Shortcode']); nps['date'] = pd.to_datetime(nps['Start time'], errors='coerce')
nps = nps.dropna(subset=['sc', 'date']).sort_values('date')
def latest_nps(c, me):
    d = nps[(nps['sc'] == c) & (nps['date'] <= me) & (nps['date'] > me - pd.Timedelta(days=365))]
    return d['Score'].iloc[-1] if len(d) else 'No response'
r_['nps'] = [latest_nps(c, me) for c, me in zip(r_['client'].str.upper(), r_['month_end'])]

ends = {'UB4108': ub.loc[real, 'date'].max(), 'NCNS': ncns['Date'].max(), 'Tardiness': tard['Date'].max(), 'Absences': absn['Date'].max(),
        'Late payments': lp['date'].max(), 'Client NPS': nps['date'].max()}
print(f"2023+ layer: {len(r_):,} snapshots from {r_.client.nunique():,} clients ({r_.month.min()} to {r_.month.max()}); "
      f"{r_.y.mean():.1%} reach {TARGET}+ within {H_RECENT} months ({int(r_.groupby('client').y.max().sum())} clients).")
print('Last date in each source: ' + ', '.join(f"{k} {v:%b %Y}" for k, v in ends.items()))
r_[['ub_resignation_12m', 'ub_client_exit_12m', 'ub_replacement_12m', 'ncns_6m', 'tardiness_6m', 'absence_6m', 'invoiced_6m', 'late_payment_6m']].mean().map('{:.0%}'.format).rename('share of snapshots with the signal').to_frame()

In [ ]:
# 13 · 2023+ layer: signal table (24-month outcome)
F2 = {
    'Exit reasons (UB4108, 12 m)': {
        'A staff resignation': r_['ub_resignation_12m'], 'No staff resignation': ~r_['ub_resignation_12m'],
        'A client-initiated exit': r_['ub_client_exit_12m'], 'An exit with a replacement requested': r_['ub_replacement_12m'],
    },
    'Attendance (6 m)': {
        'Any no call no show': r_['ncns_6m'], 'Any late arrival': r_['tardiness_6m'], 'Any notified absence': r_['absence_6m'],
        'No attendance record at all': ~(r_['ncns_6m'] | r_['tardiness_6m'] | r_['absence_6m']),
    },
    'Payments (6 m)': {'Invoiced, any invoice paid late': r_['late_payment_6m'], 'Invoiced, every invoice on time': r_['invoiced_6m'] & ~r_['late_payment_6m'],
                       'Not in the payment report': ~r_['invoiced_6m']},
    'Client NPS (12 m)': {f'NPS: {v}': r_['nps'] == v for v in ['Promoter', 'Passive', 'Detractor', 'No response']},
    'Same signals as above, for comparison': {
        '2 seats now (vs 1)': r_['seats'] == 2,
    },
}
sig2, base2 = signal_table(r_, F2, min_clients=10)
print(f"Typical 1-2 seat client in this window: {base2:.1%} reach {TARGET}+ within {H_RECENT} months.")
plot_signals(sig2, f'2023-24 snapshots: signals and the chance of reaching {TARGET}+ within {H_RECENT} months', base2)
show_sig(sig2)

## What the climb looks like

For the clients that did reach 5+ from a 1–2 seat snapshot: how long it took, and whether they got there in **steps** or **batches** of seats.

In [ ]:
# 14 · The path to 5+: months from the last 1-2 seat month-end to the first 5+ month-end, and seats added per month on the way
growers = s[s['y'] == 1].groupby('client').agg(last_small_t=('t', 'max'))
hit_t = p[(p.seats >= TARGET)].groupby('client')['t'].min()
growers['first5_t'] = hit_t.reindex(growers.index)
# use each grower's first 1-2 seat snapshot that led to 5+, and the last small month-end before the first 5+ month
first_small = s[s['y'] == 1].groupby('client')['t'].min()
growers['from_first_snapshot_m'] = growers['first5_t'] - first_small
last_small = p[p.seats.isin(SMALL)].merge(growers['first5_t'], left_on='client', right_index=True)
last_small = last_small[last_small['t'] < last_small['first5_t']].groupby('client')['t'].max()
growers['from_last_small_m'] = growers['first5_t'] - last_small
# largest one-month jump on the way up (batch hiring)
pw = p.merge(growers[['first5_t']], left_on='client', right_index=True)
pw = pw[(pw['t'] <= pw['first5_t'])].sort_values(['client', 't'])
pw['jump'] = pw.groupby('client')['seats'].diff()
growers['biggest_monthly_jump'] = pw.groupby('client')['jump'].max()
print(f"{len(growers)} clients went from 1-2 seats to {TARGET}+.")
desc = pd.DataFrame({
    'median': growers[['from_first_snapshot_m', 'from_last_small_m', 'biggest_monthly_jump']].median(),
    'middle half': growers[['from_first_snapshot_m', 'from_last_small_m', 'biggest_monthly_jump']].quantile([.25, .75]).T.apply(lambda r: f"{r.iloc[0]:.0f}-{r.iloc[1]:.0f}", axis=1),
}).rename(index={'from_first_snapshot_m': 'months from first 1-2 seat snapshot to 5+', 'from_last_small_m': 'months from last month at 1-2 seats to 5+',
                 'biggest_monthly_jump': 'largest seats added in one month on the way'})
display(desc)
jb = pd.cut(growers['biggest_monthly_jump'], [0, 1, 2, 3, 100], labels=['1 at a time', '2 in one month', '3 in one month', '4+ in one month']).value_counts().sort_index()
(jb / jb.sum()).map('{:.0%}'.format).rename('share of growers, by largest one-month jump').to_frame()

## Findings

**Short answer.** Going from 1–2 seats to 5+ is uncommon: **11% of 1–2 seat client-months reach 5+ within 3 years**. Over the whole period, 217 of 1,216 small clients (18%) got there. The signals that matter are about **momentum and timing**: is the client still adding seats, and is it early in its life? Who the client is (region, billing) matters little, and what the seats do matters only somewhat.

### Signals towards growth
1. **Already at 2 seats.** 20% of 2-seat snapshots reach 5+, vs 6.8% at 1 seat (about 3×). This is the largest factor in both models.
2. **Momentum.**
   - Grew in the last 6 months: 1.8×.
   - Added a seat in the last 3 months: 1.4×.
   - Exactly **2 seats added in the last 12 months**: 19%.
   - 3+ added while still at 1–2 seats means staff churning through the seats, and growth drops to 10% (3 added) and 5% (4+).
3. **An early 2nd seat.** 19–23% when the 2nd seat came within the first year, 15% in year 2 and 9% after 2+ years.
4. **A second role, not a second copy.** 2 seats in **different JP categories**: 2.0× (the client is expanding into another function). The same category is 1.5× and not clear.
   - Seat categories that lean towards growth: BPO Voice 2.1×, Administrative 1.9×, Accounting 1.4×. Sales Professional is 2.7× but not clear.
5. **2023+ sources.** Clients in the payment report that paid **late** still grew (1.2×). Late payment is not a brake on growth for small clients.

### Lack of indicators: when growth almost never happens
1. **Time runs out.** After 3 years at 1–2 seats the chance falls to 0.3× (4% at 37–60 months). **None of the 41 clients still at 1–2 seats after 5 years grew to 5+.**
2. **Stalling.**
   - No seat added in 12 months: 0.7×.
   - No seat added in 2 years: 0.5×.
   - The same seat count for 12+ months: 0.8×.
3. **Staff exits and shrinking.**
   - Any staff exit in the last 12 months: 0.4×.
   - An exit that wasn't replaced: 0.3×.
   - Even a replaced exit: 0.5×.
   - Shrank in the last 6 months: 0.4×.
   - Was bigger before (peak of 3–4 seats): 0.5×.
4. **Long-serving staff (24+ months): 0.5× on its own, but this mostly reflects client age.** With client age held constant, the logistic model gives stable staff a *positive* weight. The brake is a client that has stopped changing, not staff who stay.
5. **2023+ sources.**
   - A **client-initiated exit** in the last 12 months: 0.4×.
   - **Client NPS passive or detractor**: 2 of 74 and 0 of 24 grew.
   - **Promoters grew no more than typical** (1.0×). A happy client is not, by itself, a growing one.
6. **Roles that lean against growth.** Real Estate 0.4×, Legal Specialist 0.2× (1 of 15 clients grew). Virtual Assistant is 0.7× and not clear.

### Not signals either way
- **Region and billing currency**, compared among clients in MyStaff.
- **Most industries.** Professional Advisory is the exception: 2.6×, 10 of 19 clients, a small group.
- **Attendance**: no call no shows, late arrivals and absences.
- **Starting with 2+ seats in the first month** (1.3×, not clear).
- Whether the seat's **role is unknown**.

### How predictable is it?
- All signals together reach **AUC 0.73** (logistic regression), both on clients the model never saw and on later years (2022–23). That is useful for ranking, not for calling individual clients.
- The **top 10%** of snapshots by LightGBM score grow at 21% (1.9× typical), and the bottom half at 6%.
- Most of what decides growth, such as the client's own business plans, isn't in this data.

### What the climb looks like
- Growers took a median **16 months** from their first 1–2 seat snapshot to 5+.
- Once growth started it was quick: a median **7 months** from the last month at 1–2 seats to 5+ (middle half 3–13).
- Most added seats one or two at a time: the largest one-month jump was 1 seat for 35% of growers, 2 seats for 41% and 3+ for 25%.

**Practical reading.** The window is the **first 2–3 years**. A small client that adds a 2nd seat early, in a different function, and keeps adding without losing staff is the one to back. A client that has sat at 1–2 seats for 2+ years with no additions, or has had a staff exit or a client-initiated exit, has very likely reached its size.

# Part 2 · From 1 seat to 2: what comes before the first step?

Part 1 found that the strongest precursors of 5+ are **being at 2 seats** and **momentum** (recent additions). This part goes one step earlier: what comes before a 1-seat client adds its **first** extra seat?

**Snapshot** = a client at a month-end with **exactly 1 seat** that has **never had 2+ seats** (from Jan 2018).
**Outcome** = **2+ seats at two consecutive month-ends** within the next **12 months**. The two month-ends rule out handovers, where a replacement starts before the first person leaves; about 5% of first 2-seat months fall straight back to 1.

New signals here are about **the one person in the seat**, built from the engagement list (leak-free, as of the snapshot):
- **Months the current person has been in the seat**, and whether they are the **original** hire or a replacement.
- **How many people** have held the seat before.
- Whether the person **had worked at Cloudstaff before** (an earlier engagement at any account, including internal pools) or was new to Cloudstaff.

In [ ]:
# 16 · 1-seat snapshots and the outcome: sustained 2+ seats within 12 months
H2 = 12
q = p.copy()
gq = q.groupby('client', sort=False)
nxt_seats = gq['seats'].shift(-1)
q['sust2'] = ((q['seats'] >= 2) & (nxt_seats >= 2)).astype(float).where(nxt_seats.notna())   # last month: can't confirm
# any sustained 2+ month-end in t+1 .. t+H2
q['fut2'] = q.groupby('client', sort=False)['sust2'].transform(lambda x: x[::-1].rolling(H2, min_periods=H2).max()[::-1].shift(-1))
q['ever2'] = q.groupby('client', sort=False)['seats'].cummax()
s1_all = q[(q['seats'] == 1) & (q['ever2'] < 2) & (q['month'] >= SNAP_FROM)].copy()
s1 = s1_all[s1_all['fut2'].notna()].copy()
s1['y'] = s1['fut2'].astype(int)
s1['churned'] = [exit_within(c, t, H2) for c, t in zip(s1['client'], s1['t'])]
s1['age_m'] = s1['t'] - s1['first_t'] + 1
base1 = s1['y'].mean()
print(f"{len(s1):,} 1-seat snapshots (never had 2+ before) from {s1.client.nunique():,} clients, {s1.month.min()} to {s1.month.max()}.")
print(f"Within 12 months: {base1:.1%} added a sustained 2nd seat, {s1.churned.mean():.0%} churned out, the rest stayed at 1. "
      f"Clients: {s1.groupby('client').y.max().sum():,} of {s1.client.nunique():,} ({client_rate(s1):.0%}) made the step from one of these snapshots.")

In [ ]:
# 17 · When does the 2nd seat come? Monthly chance of adding it, by months since the first seat
#      (clients whose first seat is Jan 2018 or later; a month counts while the client is still at exactly 1 seat)
fresh = q[(q['first_t'] >= SNAP_FROM.ordinal)].copy()
fresh['age_m'] = fresh['t'] - fresh['first_t'] + 1
fresh['next_seats'] = fresh.groupby('client')['seats'].shift(-1)
fresh['next_sust2'] = fresh.groupby('client')['sust2'].shift(-1)
risk = fresh[(fresh['seats'] == 1) & (fresh['ever2'] < 2) & fresh['next_sust2'].notna()]
haz = risk.groupby('age_m').agg(at_risk=('client', 'size'), added=('next_sust2', 'sum'),
                                lost=('next_seats', lambda x: (x == 0).sum()))
haz = haz[haz['at_risk'] >= 50]
haz['add_rate'] = haz['added'] / haz['at_risk']; haz['loss_rate'] = haz['lost'] / haz['at_risk']
fig, ax = plt.subplots(figsize=(9, 3.8))
sm = lambda x: x.rolling(3, center=True, min_periods=1).mean()
ax.plot(haz.index, sm(haz['add_rate']), color=C_UP, lw=2, label='adds a 2nd seat next month')
ax.plot(haz.index, sm(haz['loss_rate']), color=C_DOWN, lw=1.5, ls='--', label='drops to 0 next month')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}')); ax.grid(alpha=.3); ax.set_ylim(0, None)
ax.set_xlabel('Months since the first seat (client still at exactly 1 seat)'); ax.set_ylabel('chance in the next month')
ax.set_title('1-seat clients: the monthly chance of adding a 2nd seat vs leaving (3-month smoothed)'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
bands = pd.cut(haz.index.to_series(), [0, 3, 6, 12, 24, 36, 10**4], labels=['1-3', '4-6', '7-12', '13-24', '25-36', '37+'])
hb = haz.groupby(bands.values, observed=True)[['at_risk', 'added', 'lost']].sum()
hb['monthly chance of a 2nd seat'] = (hb['added'] / hb['at_risk']).map('{:.1%}'.format)
hb['monthly chance of leaving'] = (hb['lost'] / hb['at_risk']).map('{:.1%}'.format)
hb.index.name = 'months since first seat'
hb.rename(columns={'at_risk': 'client-months at 1 seat', 'added': '2nd seat added', 'lost': 'dropped to 0'})

In [ ]:
# 18 · The person in the seat, at each 1-seat snapshot (engagement list only, as of the snapshot)
s1['month_end'] = s1['month'].dt.to_timestamp(how='end').dt.normalize()
cur_seat = s1[['client', 'month_end']].reset_index().merge(eng[['client', 'Engagement ID', 'CSID', 'start', 'end']], on='client')
cur_seat = cur_seat[(cur_seat['start'] <= cur_seat['month_end']) & (cur_seat['end'].isna() | (cur_seat['end'] >= cur_seat['month_end']))]
assert not cur_seat['index'].duplicated().any() and len(cur_seat) == len(s1)
cur_seat = cur_seat.set_index('index').reindex(s1.index)
s1['person_tenure_m'] = (s1['month_end'] - cur_seat['start']).dt.days / 30.44
first_start = eng.groupby('client')['start'].min()
s1['original_person'] = cur_seat['start'].eq(s1['client'].map(first_start))
starts_by_client = eng.groupby('client')['start'].apply(lambda x: np.sort(x.to_numpy()))
s1['people_before'] = [int((starts_by_client[c] < np.datetime64(st)).sum()) for c, st in zip(s1['client'], cur_seat['start'])]
# worked at Cloudstaff before this seat: any earlier engagement for the same CSID, any account (internal pools included)
all_eng = raw[['CSID', 'Engagement Date']].assign(d=lambda d: pd.to_datetime(d['Engagement Date'], errors='coerce')).dropna(subset=['CSID', 'd'])
first_cs = all_eng.groupby('CSID')['d'].min()
s1['experienced_person'] = cur_seat['CSID'].map(first_cs).lt(cur_seat['start'] - pd.Timedelta(days=14))
s1['cat'] = cur_seat['CSID'].map(by_csid['JP Category (Filled)']).fillna('Unknown')
s1['prior_churn'] = [bool((exit_t.get(c) is not None) and (exit_t[c] < t).any()) for c, t in zip(s1['client'], s1['t'])]
s1['snap_year'] = s1['month'].dt.year
print(f"Person in the seat: original hire in {s1.original_person.mean():.0%} of snapshots, worked at Cloudstaff before in {s1.experienced_person.mean():.0%}, "
      f"median {s1.person_tenure_m.median():.0f} months in the seat.")
s1[['age_m', 'person_tenure_m', 'people_before']].describe().T.round(1)

In [ ]:
# 19 · Signal table: "x as likely to add a sustained 2nd seat within 12 months as the typical 1-seat client"
cat1 = s1['cat'].value_counts(); cats1 = [c for c in cat1.index if c != 'Unknown' and cat1[c] >= 300]
F1 = {
    'Client stage': {
        'First seat under 6 months ago': s1['age_m'] <= 6,
        'First seat 7-12 months ago': s1['age_m'].between(7, 12),
        'First seat 13-24 months ago': s1['age_m'].between(13, 24),
        'First seat 2+ years ago': s1['age_m'] > 24,
        'Came back after an earlier churn-out': s1['prior_churn'],
    },
    'The person in the seat': {
        'Person in the seat under 3 months': s1['person_tenure_m'] < 3,
        'Person in the seat 3-6 months': s1['person_tenure_m'].between(3, 6),
        'Person in the seat 6-12 months': (s1['person_tenure_m'] > 6) & (s1['person_tenure_m'] <= 12),
        'Person in the seat 12+ months': s1['person_tenure_m'] > 12,
        'Still the original hire': s1['original_person'],
        'A replacement (seat refilled at least once)': ~s1['original_person'],
        'Seat refilled 2+ times': s1['people_before'] >= 2,
        'Person worked at Cloudstaff before': s1['experienced_person'],
        'Person new to Cloudstaff': ~s1['experienced_person'],
    },
    'Role of the seat': {**{f'Seat in {c}': s1['cat'] == c for c in cats1}, 'Seat role unknown': s1['cat'] == 'Unknown'},
}
sig1, b1 = signal_table(s1, F1)
print(f"Typical 1-seat client: {b1:.1%} add a sustained 2nd seat within {H2} months.")
plot_signals(sig1, f'1-seat snapshots: signals and the chance of a sustained 2nd seat within {H2} months', b1)
show_sig(sig1)

In [ ]:
# 20 · Client age vs the person's time in the seat: which one matters? (rate of a 2nd seat within 12 months)
#      They move together for the original hire; replacements break the link.
s1['client age'] = pd.cut(s1['age_m'], [0, 6, 12, 24, 10**4], labels=['0-6 m', '7-12 m', '13-24 m', '24+ m'])
s1['person in seat'] = pd.cut(s1['person_tenure_m'], [-1, 3, 6, 12, 10**4], labels=['under 3 m', '3-6 m', '6-12 m', '12+ m'])
def cell(g): return f"{g.y.mean():.0%} ({g.client.nunique()} cl.)" if g.client.nunique() >= 15 else '–'
grid = s1.groupby(['client age', 'person in seat'], observed=True).apply(cell, include_groups=False).unstack()
display(Markdown("**All 1-seat snapshots** (rows: client age; columns: months the current person has been in the seat; cells with under 15 clients hidden)"))
display(grid)
grid_r = s1[~s1.original_person].groupby(['client age', 'person in seat'], observed=True).apply(cell, include_groups=False).unstack()
display(Markdown("**Replacement in the seat only** (the client is older than the person's tenure)"))
display(grid_r)

# logistic model with both, plus the other person signals: weights per 1 sd, client-grouped CV AUC
d = s1.assign(log_age=np.log(s1.age_m), log_ten=np.log1p(s1.person_tenure_m.clip(lower=0)),
              replacement=(~s1.original_person).astype(int), experienced=s1.experienced_person.astype(int), refills=s1.people_before.clip(upper=4))
cols = ['log_age', 'log_ten', 'replacement', 'experienced']        # 'times refilled' left out: almost the same as 'replacement'
lr1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
oof1 = np.zeros(len(d))
for tr, te in GroupKFold(5).split(d, d.y, d.client):
    oof1[te] = lr1.fit(d[cols].iloc[tr], d.y.iloc[tr]).predict_proba(d[cols].iloc[te])[:, 1]
lr1.fit(d[cols], d.y)
w = pd.Series(lr1[-1].coef_[0], index=['Client age (log months)', 'Person in seat (log months)', 'Person is a replacement', 'Person worked at Cloudstaff before'])
print(f"Logistic model on the person and stage signals: client-grouped CV AUC {roc_auc_score(d.y, oof1):.3f}. Weights per 1 sd (positive = more likely to add a 2nd seat):")
w.round(2).rename('weight').to_frame()

In [ ]:
# 21 · 2023+ sources for 1-seat clients: attendance, exits, payments, NPS (snapshots Jan 2023 - Oct 2025, 12-month outcome)
r1 = s1_all[(s1_all['month'] >= pd.Period('2023-01', 'M')) & s1_all['fut2'].notna()].copy()
r1['y'] = r1['fut2'].astype(int)
r1['month_end'] = r1['month'].dt.to_timestamp(how='end').dt.normalize()
r1['ub_resignation_12m'] = recent_any(r1, ub, 'date', 12, real & ub['Reason for Disengagement'].astype('string').eq('Voluntary'))
r1['ub_client_exit_12m'] = recent_any(r1, ub, 'date', 12, real & ub['Type of Involuntary'].astype('string').eq('Customer Initiated'))
r1['ncns_6m'] = recent_any(r1, ncns, 'Date', 6); r1['tardiness_6m'] = recent_any(r1, tard, 'Date', 6); r1['absence_6m'] = recent_any(r1, absn, 'Date', 6)
r1['invoiced_6m'] = recent_any(r1, lp, 'date', 6); r1['late_payment_6m'] = recent_any(r1, lp, 'date', 6, lp['late_flag'] == 1)
r1['nps'] = [latest_nps(c, me) for c, me in zip(r1['client'].str.upper(), r1['month_end'])]
# Absences stop on 16 Feb 2026: attendance signals only for snapshots it covers
att = r1[r1['month_end'] <= pd.Timestamp('2026-02-16')]
F1b = {
    'Exit reasons (UB4108, 12 m)': {'A staff resignation': r1['ub_resignation_12m'], 'A client-initiated exit': r1['ub_client_exit_12m']},
    'Payments (6 m)': {'Invoiced, any invoice paid late': r1['late_payment_6m'], 'Invoiced, every invoice on time': r1['invoiced_6m'] & ~r1['late_payment_6m'],
                       'Not in the payment report': ~r1['invoiced_6m']},
    'Client NPS (12 m)': {f'NPS: {v}': r1['nps'] == v for v in ['Promoter', 'Passive', 'Detractor', 'No response']},
}
F1c = {'Attendance of the seat (6 m)': {'Any no call no show': att['ncns_6m'], 'Any late arrival': att['tardiness_6m'], 'Any notified absence': att['absence_6m'],
                                        'No attendance record at all': ~(att['ncns_6m'] | att['tardiness_6m'] | att['absence_6m'])}}
sig1b, b1b = signal_table(r1, F1b, min_clients=10)
sig1c, b1c = signal_table(att, F1c, min_clients=10)
sig1c['x typical'] = sig1c['x typical'] * b1c / b1b                          # re-express against the same typical rate
sig1c[['lo', 'hi']] = sig1c[['lo', 'hi']] * b1c / b1b
both = pd.concat([sig1b, sig1c], ignore_index=True)
print(f"{len(r1):,} 1-seat snapshots from {r1.client.nunique():,} clients ({r1.month.min()} to {r1.month.max()}); typical: {b1b:.1%} add a sustained 2nd seat within {H2} months. "
      f"Attendance rows use the {len(att):,} snapshots up to Feb 2026.")
plot_signals(both, f'2023-25 1-seat snapshots: signals and the chance of a 2nd seat within {H2} months', b1b)
show_sig(both)

In [ ]:
# 22 · What the 2nd seat looks like, and does it matter for what comes next?
#      First sustained 2-seat month-end: is the 2nd person in the same JP category as the first, or a new role?
prev_ever = q.groupby('client')['ever2'].shift().fillna(0)
prev_seats = q.groupby('client')['seats'].shift()
# a true first step: 1 seat the month before, exactly 2 now, and still 2+ the month after
step = q[(q['sust2'] == 1) & (prev_ever < 2) & (prev_seats == 1) & (q['seats'] == 2) & (q['month'] >= SNAP_FROM)].groupby('client').head(1)[['client', 't', 'month']]
step['month_end'] = step['month'].dt.to_timestamp(how='end').dt.normalize()
pair = step.merge(eng[['client', 'CSID', 'start', 'end']], on='client')
pair = pair[(pair['start'] <= pair['month_end']) & (pair['end'].isna() | (pair['end'] >= pair['month_end']))].sort_values('start')
pair['cat'] = pair['CSID'].map(by_csid['JP Category (Filled)']).fillna('Unknown')
pp2 = pair.groupby('client').agg(cats=('cat', list), n=('cat', 'size'))
pp2 = pp2[pp2['n'] >= 2]
def kind(c):
    if 'Unknown' in c[:2]: return 'role unknown'
    return 'same JP category as the first' if c[0] == c[1] else 'a different JP category'
pp2['2nd seat'] = pp2['cats'].map(kind)
pp2 = pp2.join(step.set_index('client')[['t']])
fm = p.set_index(['client', 't'])['seats']
def reached(c, t, k, h):
    v = fm.reindex([(c, x) for x in range(t + 1, t + h + 1)]).dropna()
    return float((v >= k).any()) if len(v) == h else np.nan
pp2['r3'] = [reached(c, t, 3, 24) for c, t in zip(pp2.index, pp2['t'])]
pp2['r5'] = [reached(c, t, 5, 36) for c, t in zip(pp2.index, pp2['t'])]
out = pp2.groupby('2nd seat').agg(clients=('n', 'size'), r3=('r3', 'mean'), n3=('r3', 'count'), r5=('r5', 'mean'), n5=('r5', 'count'))
out['share'] = (out['clients'] / out['clients'].sum()).map('{:.0%}'.format)
out['reached 3+ within 24 m'] = out.apply(lambda r: f"{r.r3:.0%} (of {int(r.n3)})", axis=1)
out['reached 5+ within 36 m'] = out.apply(lambda r: f"{r.r5:.0%} (of {int(r.n5)})", axis=1)

print(f"{len(pp2):,} clients made a sustained step from exactly 1 to exactly 2 seats from {SNAP_FROM} on. Outcomes use only clients with enough months after the step.")
out[['clients', 'share', 'reached 3+ within 24 m', 'reached 5+ within 36 m']]

## Findings · Part 2 (1 → 2 seats)

**Short answer.** The precursor to momentum is mostly **timing**. The 2nd seat comes early or rarely. Very little about the one person in the seat predicts it. The seat's role and some 2023+ service signals tilt the odds a little.

1. **How often.** 21% of 1-seat client-months lead to a sustained 2nd seat within 12 months, and 32% lead to churning out. Overall, 43% of 1-seat clients made the step at some point.
2. **When.** The monthly chance of adding a 2nd seat is highest at the start and **falls steadily**:

   | Months since the first seat | Monthly chance of a 2nd seat |
   |---|---|
   | 1–3 | 7.5% |
   | 4–6 | 4.2% |
   | 7–12 | 3.2% |
   | Year 2 | 2.1% |
   | Year 3 | 1.4% |
   | After that | about 1% |

   There is **no bump at the 6-month probation point or at the anniversary**. From about month 3, a 1-seat client is more likely to **leave than to add** in any given month. Some very early 2nd seats are probably part of the original order, starting a few weeks later; the data can't separate those.
3. **The person in the seat is mostly not a signal.**
   - The person's time in the seat looks like one (under 3 months 1.4×, 12+ months 0.6×), but it follows the **client's age**. At the same client age, the rate barely moves with the person's tenure: 17–18% across the board at 13–24 months.
   - Original hire vs replacement: 0.8×, not clear.
   - Person had worked at Cloudstaff before: 1.0×.
   - A model on these signals and client age only reaches **AUC 0.62**.
4. **Role of the seat matters a little here.** It did not matter for 1-seat churn (`stable_seat_churn.ipynb` Part 3).
   - **IT Software Development** 1.8× (39% add a 2nd seat) and **Sales Professional** 1.7× (35%).
   - **Real Estate** 0.5×.
   - Most other categories are about 1×.
5. **2023+ sources** (snapshots Jan 2023 – Sep 2025):
   - Clients that paid **every invoice on time**: 1.3× (23%). Late payers 1.0×. Not in the payment report: 0.6×.
   - **Late arrivals** in the last 6 months: 0.8×. No call no shows: 0.7×, not clear.
   - **Staff resignation** in the last 12 months: 0.4×.
   - **NPS passive**: 0.3× (3 of 35 clients). **Promoters** are no more likely to add a seat (0.7×, not clear). A satisfied 1-seat client is not, by itself, an expanding one.
6. **What happens after the step.** 575 clients made a clean step from exactly 1 to exactly 2 seats.
   - **52–63%** reached 3+ within 24 months. Once a client takes the first step, the next one is likely.
   - The 2nd seat was a **different JP category** in 35% of cases and the same category in 26% (39% unknown).
   - A different-category 2nd seat reached 5+ within 36 months at 30%, vs 22% for the same category. That matches Part 1, but the numbers are small (101 and 72 clients).

**The chain, end to end.**
- An early 2nd seat (months 1–6) → 3+ seats (more than half of those that step) → 5+ (more than a third of those at 2 seats within the first year, Part 1).
- The best precursor of momentum is **earlier momentum**. The best time to win the 2nd seat is the **first 6 months**.
- Within that window, the odds tilt up with a **tech or sales role**, **on-time payments** and **clean attendance**. They tilt down with a **staff resignation**, **late arrivals** or a **lukewarm (passive) NPS**.
- A client that has sat at 1 seat for 18+ months is unlikely to step up: about 1–2% a month.

# Part 3 · How big do clients start?

**Question.** What share of new clients start with 1 seat, 2–4 seats, 5+ seats, and so on?

**Starting size** = the client's seat count at the **end of its second month**: the month of its first seat, plus a **1-month ramp-up**. Seats that start a few weeks after the first one count towards the starting size; anything later is growth. Clients that are already back to 0 at that month-end are shown as their own group. Step 24 compares this with no ramp-up and with a 2-month ramp-up.

**Clients.** External clients whose **first seat is in Jan 2016 or later**, with that second month-end in the data (first seat by Sep 2026). A client that left and came back is counted once, by its first start.

In [ ]:
# 23 · Starting size after a 1-month ramp-up
RAMP = 1                                         # months after the first-seat month
START_FROM = pd.Period('2016-01', 'M')
START_BANDS = [(0, 0, 'back to 0 by then'), (1, 1, '1'), (2, 2, '2'), (3, 4, '3-4'), (5, 9, '5-9'), (10, 19, '10-19'), (20, 10**6, '20+')]
BAND_ORDER = [b[2] for b in START_BANDS]
def start_band(n):
    return next(lbl for lo, hi, lbl in START_BANDS if lo <= n <= hi)

pa = p[['client', 't', 'month', 'seats', 'first_t']].copy()
pa['age_m'] = pa['t'] - pa['first_t'] + 1                # 1 = month of the first seat
first_month = pa.groupby('client')['first_t'].first()
never_at_month_end = first_month.isna().sum()          # every engagement started and ended inside one month
first_month = first_month.dropna().astype(int)
new_clients = first_month[(first_month >= START_FROM.ordinal) & (first_month + RAMP <= LAST_MONTH.ordinal)].index
seats_at = pa[pa['age_m'] >= 1].astype({'age_m': int}).set_index(['client', 'age_m'])['seats']
def size_at(clients, age):
    return pd.Series(seats_at.reindex([(c, age) for c in clients]).to_numpy(), index=clients)

starts = pd.DataFrame({'first_month': first_month[new_clients].map(lambda o: pd.Period(ordinal=o, freq='M'))})
starts['seats_m1'] = size_at(new_clients, 1)
starts['start_seats'] = size_at(new_clients, 1 + RAMP)
starts['band'] = pd.Categorical(starts['start_seats'].map(start_band), BAND_ORDER, ordered=True)
starts['year'] = starts['first_month'].dt.year

dist = starts['band'].value_counts().reindex(BAND_ORDER).rename('clients').to_frame()
dist['share'] = dist['clients'] / dist['clients'].sum()
dist['seats they started with'] = starts.groupby('band', observed=False)['start_seats'].sum().reindex(BAND_ORDER)
dist['share of starting seats'] = dist['seats they started with'] / dist['seats they started with'].sum()
print(f"{len(starts):,} new external clients with a first seat from {START_FROM} to {LAST_MONTH - RAMP}. "
      f"Median starting size {starts.loc[starts.start_seats > 0, 'start_seats'].median():.0f} seat(s); "
      f"{(starts.start_seats >= 5).mean():.0%} start with 5+. "
      f"Not counted: {never_at_month_end} clients (any year) that never held a seat at a month-end.")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
cols = ['#C7C7CC', '#9BC9FF', '#5AA8FF', '#007AFF', '#00AB9C', '#00786D', '#4121A9']
for ax, col, title in [(axes[0], 'share', 'Share of new clients'), (axes[1], 'share of starting seats', 'Share of the seats new clients started with')]:
    ax.bar(dist.index.astype(str), dist[col], color=cols)
    for i, v in enumerate(dist[col]): ax.text(i, v, f'{v:.0%}', ha='center', va='bottom', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_ylim(0, dist[col].max() * 1.18)
    ax.set_title(title); ax.set_xlabel('seats at the end of month 2 (1-month ramp-up)'); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

show = dist.copy()
show['share'] = show['share'].map('{:.0%}'.format); show['share of starting seats'] = show['share of starting seats'].map('{:.0%}'.format)
show['seats they started with'] = show['seats they started with'].astype(int)
show.index.name = 'starting size'
show

In [ ]:
# 24 · How much the ramp-up choice matters: starting size at the end of month 1 (no ramp-up), 2 (default) and 3
cmp = {}
for ramp in (0, 1, 2):
    ok = first_month[(first_month >= START_FROM.ordinal) & (first_month + ramp <= LAST_MONTH.ordinal)].index
    b = size_at(ok, 1 + ramp).map(start_band)
    cmp[f'end of month {1 + ramp}' + (' (no ramp-up)' if ramp == 0 else ' (default)' if ramp == RAMP else '')] = b.value_counts(normalize=True).reindex(BAND_ORDER).fillna(0)
cmp = pd.DataFrame(cmp)
moved = (starts['start_seats'] != starts['seats_m1'])
print(f"Between month 1 and month 2, {moved.mean():.0%} of new clients changed seat count "
      f"({(starts.start_seats > starts.seats_m1).mean():.0%} added, {(starts.start_seats < starts.seats_m1).mean():.0%} lost).")
cmp.map('{:.0%}'.format).rename_axis('starting size')

In [ ]:
# 25 · Has the starting mix changed over time? (share of each year's new clients, 1-month ramp-up)
by_y = pd.crosstab(starts['year'], starts['band'], normalize='index').reindex(columns=BAND_ORDER, fill_value=0)
n_y = starts.groupby('year').size()
fig, ax = plt.subplots(figsize=(10, 3.8))
bottom = np.zeros(len(by_y))
for c, colr in zip(BAND_ORDER, cols):
    ax.bar(by_y.index.astype(str), by_y[c], bottom=bottom, color=colr, label=c)
    for i, (b0, v) in enumerate(zip(bottom, by_y[c])):
        if v >= .07: ax.text(i, b0 + v / 2, f'{v:.0%}', ha='center', va='center', fontsize=8, color='white' if c not in ('back to 0 by then', '1', '2') else 'black')
    bottom += by_y[c].to_numpy()
ax.set_xticks(range(len(by_y)), [f'{y}\n(n={n_y[y]})' for y in by_y.index], fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_ylim(0, 1)
ax.set_title('Starting size of new clients, by year of first seat'); ax.legend(title='starting seats', frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()
t = by_y.map('{:.0%}'.format); t.insert(0, 'new clients', n_y)
t

In [ ]:
# 26 · What happens next, by starting size: churn-out and growth over the following months
#      Measured from the end of the ramp-up month; each horizon uses only clients with that much data after it.
st = starts[starts['start_seats'] > 0].copy()
st['t0'] = first_month[st.index] + RAMP
def churned_by(c, t0, h):
    a = exit_t.get(c)
    return bool(a is not None and ((a > t0) & (a <= t0 + h)).any())
rows = []
for b, gb in st.groupby('band', observed=True):
    row = {'starting size': b, 'clients': len(gb)}
    for h in (12, 24):
        ok = gb[gb['t0'] + h <= LAST_MONTH.ordinal]
        row[f'churned out within {h} m'] = np.mean([churned_by(c, t0, h) for c, t0 in zip(ok.index, ok['t0'])]) if len(ok) else np.nan
        row[f'seats at {h} m vs start: grew'] = np.mean([(seats_at.get((c, 1 + RAMP + h), np.nan) > s0) for c, s0 in zip(ok.index, ok['start_seats'])]) if len(ok) else np.nan
    ok = gb[gb['t0'] + 24 <= LAST_MONTH.ordinal]
    pk = [fm.reindex([(c, x) for x in range(t0 + 1, t0 + 25)]).max() for c, t0 in zip(ok.index, ok['t0'])]
    row['reached 5+ within 24 m'] = np.mean([v >= 5 for v in pk]) if len(ok) else np.nan
    row['clients judged at 24 m'] = len(ok)
    rows.append(row)
nxt = pd.DataFrame(rows).set_index('starting size')
for c in [c for c in nxt.columns if 'clients' not in c]:
    nxt[c] = nxt[c].map(lambda v: '–' if pd.isna(v) else f'{v:.0%}')
nxt.loc[['5-9', '10-19', '20+'], 'reached 5+ within 24 m'] = 'already 5+'
nxt

## Findings · Part 3 (starting size)

**Most clients start with 1 seat.** Of 2,215 new external clients (first seat Jan 2016 – Sep 2026), after a 1-month ramp-up:

| Starting size | Share of clients | Share of the seats new clients started with |
|---|---|---|
| 1 | **69%** | 37% |
| 2 | 15% | 16% |
| 3–4 | 7% | 13% |
| 5–9 | 3% | 11% |
| 10–19 | 2% | 12% |
| 20+ | 1% | 10% |
| back to 0 by then | 4% | – |

- **84% start with 1–2 seats, 7% with 3–4 and 6% with 5+.**
- The 6% that start with 5+ bring **a third of all starting seats**.

**The ramp-up month matters, but only at the margin.**
- With no ramp-up (end of month 1), 80% look like 1-seat clients. One month later, 14% of clients have added seats, mostly a 2nd person starting a few weeks after the first, so the 1-seat share falls to 69%.
- A 2-month ramp-up moves the 1-seat share only to 61%, while the share already gone rises to 7%. One month is a reasonable cut-off between "initial order" and "growth".

**The mix is fairly stable over time, but big starts have become rarer.**
- 1-seat starts have been 62–73% of new clients every year since 2017, at the top of that range in 2024–2026 (72–73%).
- 5+ starts were 6–10% of new clients a year from 2016 to 2023 and **2–4% in 2024–2026**.

**Starting size shapes what follows** (measured from the end of the ramp-up month):

| Starting size | Churned out within 12 m | Within 24 m | Reached 5+ within 24 m |
|---|---|---|---|
| 1 | 37% | 51% | 7% |
| 2 | 21% | 41% | 22% |
| 3–4 | 14% | 28% | 56% |
| 5+ | 13–25% | 31–36% | already 5+ |

- Half of 1-seat starters are gone within 2 years, and few become 5+.
- A start at 3–4 seats makes 5+ the likely outcome (56%).
- 20+ starts are only 12 clients, too few to read on their own.

## Assumptions

| # | Assumption | Why |
|---|---|---|
| 1 | **Seat**, **client**, internal-account removal and the cleaning steps are the same as in `stable_seat_churn.ipynb` (steps 2–5 there). A seat counts at a month-end if the engagement is active that day. | One set of rules across the churn and growth notebooks. |
| 2 | **Snapshot** = a client at a month-end with **1 or 2 seats**, from **Jan 2018**, that has **never had 5+ seats** up to that month-end. | The question is about small clients becoming big. Shrinking former 5+ clients are a different story; step 4 shows how many there are. |
| 3 | **Grew** = **5+ seats at any month-end in the next 36 months**. Snapshots without 36 months of data after them (after Oct 2023) are left out. Reached 3+ and churned out are shown alongside. | A peak of 5+ counts, even if the client later shrinks: the question is whether it grew. 36 months, because earlier work showed growth to 5 takes about 2.5 years. |
| 4 | One client gives many monthly snapshots. Intervals resample **clients** (300 draws), cross-validation splits **by client**, and the out-of-time test splits by year. | Rows from the same client are not independent; treating them as independent would overstate certainty. |
| 5 | **Replacement** = another engagement at the same client starting within 62 days before or after a staff exit. | The engagement list has no replacement field before UB4108. |
| 6 | **Roles** come from the Staff Central staff report via `shared/jp_titles.py`. They are matched by CSID, so they show the person's role **today**. **Unknown** = no JP data even after backfill. | Same source and caveats as `stable_seat_churn.ipynb` Part 3. Unknown is tied to people who have left, so it is partly an outcome signal. |
| 7 | **MyStaff** attributes (region, industry, billing currency) are as of 11 Sep 2026 and are compared **only among clients that are in MyStaff** (step 9b). They are left out of the main table and the models. | MyStaff holds 26% of the snapshot clients: 53% of those that grew but 10% of those that churned out. "In MyStaff" mostly means "still a client today", which would leak the outcome. |
| 8 | **2023+ sources** use snapshots from Jan 2023 to Oct 2024 and a **24-month** outcome. They are joined by shortcode and month: UB4108 by termination notice date (submitted date if blank; cancelled and record-only tickets dropped), NCNS / Tardiness / Absences by incident date, Late Payments by invoice month (late flag = 1), and Client NPS as the latest response in the previous 12 months. | Their history is too short for the 36-month outcome. Joining by shortcode (not engagement) is simpler, and it is enough for "did the client have any". |
| 9 | Signals are shown when at least 15 clients have them (10 in the 2023+ layer). A signal reads **towards** or **against** growth only when its whole 95% interval is above or below 1×. | Small groups give unstable rates. |
| 10 | Models: logistic regression (standardised, balanced class weights) and LightGBM (300 trees, depth limited by 15 leaves, 60+ snapshots per leaf). Snapshot year is left out of the features; the out-of-time test trains on 2018–2021 and tests on 2022–2023. | Readable weights from one model and interactions from the other; the year split checks the result holds over time. |
| 11 | **Part 2 snapshot** = a month-end with exactly 1 seat, from Jan 2018, for a client that never had 2+ seats. **Outcome** = 2+ seats at **two consecutive month-ends** within the next 12 months. | The first step only. Requiring two month-ends rules out backfill handovers (about 5% of first 2-seat months fall straight back). |
| 12 | **Person in the seat** = the engagement active at the snapshot month-end. "Original hire" = it started on the client's first engagement date. "Worked at Cloudstaff before" = the same CSID has an engagement at any account (including internal pools) starting 14+ days earlier. | Built from the engagement list as of the snapshot, so nothing from the future leaks in. |
| 13 | Part 2's 2023+ layer uses snapshots from Jan 2023 to Oct 2025. Attendance signals are judged only up to 16 Feb 2026, the last date in Absences. | A 12-month outcome allows a longer window than Part 1's 24 months. |
| 14 | **Starting size** = seats at the end of the client's 2nd month (the first-seat month plus a 1-month ramp-up), for clients whose first seat is Jan 2016 to Sep 2026. A client that left and came back is counted once, by its first start. "Back to 0 by then" = no seats at that month-end. | Seats that start within weeks of the first one are part of the initial order, not growth. Step 24 shows the result with no ramp-up and with 2 months. |